# Biohub - Cell Tracking During Development: Boundary-Aware Hybrid Candidate Generator

The GT-centered failure analysis (`gt_failure_analysis.py`) found that even at
the most permissive achievable settings, 30/174 GT nodes (17.2%) have **no**
raw candidate within 7um - candidate generation itself, not thresholding or
NMS, is the bottleneck. Many of those nodes are near the volume's z/y/x
boundaries.

**Audit (requirement 2) before writing new code:**
- `scipy.ndimage.gaussian_filter`'s default `mode` is `'reflect'` (mirror
  edge-replication), not zero-padding - confirmed via
  `inspect.signature(gaussian_filter)`.
- `skimage.feature.peak_local_max`'s internal peak mask calls
  `scipy.ndimage.maximum_filter(image, footprint=footprint, mode='nearest')`
  - confirmed via `inspect.getsource`. `exclude_border` only drops peaks
  from the *output*, it never prevented the maximum filter from "seeing" a
  boundary peak in the first place, and the existing method A already
  passes `exclude_border=False`.
- **Conclusion:** the 30 lost nodes are not a padding-mode bug. They likely
  fall into two categories: (a) a genuine boundary local maximum that some
  other footprint/edge-case subtlety still misses (addressed by **D**, an
  independently re-implemented, explicitly-padded, unit-tested local-max
  detector), or (b) no interior local maximum exists at all - e.g. a dim
  cell dominated by a brighter neighbor, or a truncated boundary cell with
  a monotonic (non-peaked) visible profile (addressed by **E**, which ranks
  by raw intensity without requiring strict local-maximum status).

This notebook:

- **Requirement 1** - diagnoses every lost GT node: boundary-distance
  diagnostics, `lost_raw_gt_nodes.csv`, `boundary_failure_summary.csv`, and
  example patch figures.
- **Requirement 2** - the padding/border audit above, plus 6 unit tests
  confirming local maxima ARE detected at every volume face (z=0, z=max,
  y=0, y=max, x=0, x=max) for both A and the new D.
- **Requirement 3** - implements **D** (boundary-aware 3D local maxima),
  **E** (dense intensity ranking + physical NMS, no local-maximum
  requirement), and **F** (hybrid union of A+D+E, deduplicated + re-ranked).
- **Requirement 4** - evaluates A/D/E/F on the first 3 real train samples:
  `raw_candidate_coverage`, `threshold_coverage`, `nms_coverage`,
  `cap_coverage`, `avg_pred_nodes`, `recall_on_sparse_gt`,
  `mean_matched_distance_um`, `elapsed_seconds` -> `boundary_detector_comparison.csv`.
- **Requirement 5** - checks each method against the goals
  (`raw_candidate_coverage>=0.95`, `cap_coverage>=0.90`), preferring the
  lowest `avg_pred_nodes` among methods that pass both.

No tracking, no final submission - candidate generation diagnostics only.


## 1. Imports & configuration

In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import time
from pathlib import Path
from typing import Sequence

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from scipy.ndimage import gaussian_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage.feature import peak_local_max

try:
    import blosc2
except ImportError:
    blosc2 = None

try:
    import zstandard
except ImportError:
    zstandard = None

# zarr v3 `data_type` -> numpy dtype code (byte order is applied separately,
# based on the "bytes" codec's "endian" configuration).
DTYPE_CODES = {
    "bool": "b1",
    "int8": "i1", "int16": "i2", "int32": "i4", "int64": "i8",
    "uint8": "u1", "uint16": "u2", "uint32": "u4", "uint64": "u8",
    "float16": "f2", "float32": "f4", "float64": "f8",
}

NODE_PROP_NAMES = ("t", "z", "y", "x")

# Physical voxel scale for this competition's volumes (µm/voxel).
VOXEL_SIZE_UM = {"z": 1.625, "y": 0.40625, "x": 0.40625}
_VOXEL_SCALE = np.array([VOXEL_SIZE_UM["z"], VOXEL_SIZE_UM["y"], VOXEL_SIZE_UM["x"]])

## 2. Manual offline Zarr v3 reader (metadata parsing)

In [ ]:
# 1. zarr.json metadata parsing (manual offline reader, no zarr/numcodecs)
# --------------------------------------------------------------------------- #
def read_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def parse_array_metadata(array_dir: Path) -> dict:
    """Parse a zarr v3 array's `zarr.json` into shape/dtype/chunking/codecs."""
    meta_path = array_dir / "zarr.json"
    doc = read_json(meta_path)

    if doc.get("zarr_format") != 3:
        raise ValueError(f"{meta_path}: only zarr v3 is supported, got zarr_format={doc.get('zarr_format')!r}")
    if doc.get("node_type") != "array":
        raise ValueError(f"{meta_path}: not a zarr array (node_type={doc.get('node_type')!r})")

    shape = tuple(int(s) for s in doc["shape"])

    data_type = doc["data_type"]
    if isinstance(data_type, dict):  # some writers nest it as {"name": "uint16"}
        data_type = data_type.get("name")
    code = DTYPE_CODES.get(data_type)
    if code is None:
        raise ValueError(f"{meta_path}: unsupported data_type {data_type!r}")

    codecs = doc.get("codecs", [])
    endian = "little"
    for codec in codecs:
        if codec.get("name") in ("bytes", "endian"):
            endian = codec.get("configuration", {}).get("endian", "little")
            break
    dtype = np.dtype("bool") if code == "b1" else np.dtype(("<" if endian == "little" else ">") + code)

    chunk_grid = doc.get("chunk_grid", {})
    if chunk_grid.get("name", "regular") != "regular":
        raise NotImplementedError(f"{meta_path}: unsupported chunk_grid {chunk_grid.get('name')!r}")
    chunk_shape = tuple(int(c) for c in chunk_grid.get("configuration", {}).get("chunk_shape", shape))

    chunk_key_encoding = doc.get(
        "chunk_key_encoding", {"name": "default", "configuration": {"separator": "/"}}
    )
    fill_value = doc.get("fill_value", 0) or 0

    return {
        "path": array_dir,
        "shape": shape,
        "dtype": dtype,
        "chunk_shape": chunk_shape,
        "codecs": codecs,
        "chunk_key_encoding": chunk_key_encoding,
        "fill_value": fill_value,
    }


# --------------------------------------------------------------------------- #

## 3. Chunk key construction + codec decoding + whole-array/timepoint readers

In [ ]:
# 2. Chunk key construction + codec decoding
# --------------------------------------------------------------------------- #
def build_chunk_path(array_dir: Path, index: Sequence[int], chunk_key_encoding: dict) -> Path:
    """Build the on-disk chunk file path for a given chunk index tuple."""
    name = chunk_key_encoding.get("name", "default")
    separator = chunk_key_encoding.get("configuration", {}).get("separator", "/")
    index_strs = [str(i) for i in index]

    if name == "v2":
        key = separator.join(index_strs) if index_strs else "0"
    else:
        key = separator.join(["c", *index_strs]) if index_strs else "c"

    if separator == "/":
        return array_dir.joinpath(*key.split("/"))
    return array_dir / key


def decode_bytes_codecs(raw: bytes, codecs: list[dict], expected_nbytes: int) -> bytes:
    """Reverse the bytes->bytes compressor chain, then hand back raw array bytes.

    `codecs[0]` is the array->bytes codec (must be "bytes"/"endian" - a plain
    passthrough once endianness is known); `codecs[1:]` are bytes->bytes
    compressors applied in encode order, so they're undone in reverse order.
    """
    if not codecs:
        return raw

    array_bytes_codec = codecs[0]
    if array_bytes_codec.get("name") not in ("bytes", "endian"):
        raise NotImplementedError(
            f"unsupported array->bytes codec: {array_bytes_codec.get('name')!r} "
            "(only the plain 'bytes' codec is supported - no sharding)"
        )

    data = raw
    for codec in reversed(codecs[1:]):
        codec_name = codec.get("name")
        if codec_name == "blosc":
            if blosc2 is None:
                raise RuntimeError("chunk uses the 'blosc' codec but the blosc2 package is not installed")
            data = blosc2.decompress(data)
        elif codec_name == "zstd":
            if zstandard is None:
                raise RuntimeError("chunk uses the 'zstd' codec but the zstandard package is not installed")
            data = zstandard.ZstdDecompressor().decompress(data, max_output_size=expected_nbytes)
        else:
            raise NotImplementedError(f"unsupported bytes->bytes codec: {codec_name!r}")

    return data


def decode_chunk_bytes(raw: bytes, codecs: list[dict], dtype: np.dtype, chunk_shape: tuple[int, ...]) -> np.ndarray:
    count = int(np.prod(chunk_shape)) if chunk_shape else 1
    expected_nbytes = count * dtype.itemsize
    decoded = decode_bytes_codecs(raw, codecs, expected_nbytes)
    arr = np.frombuffer(decoded, dtype=dtype, count=count)
    return arr.reshape(chunk_shape)


def read_chunk_at(meta: dict, index: Sequence[int]) -> np.ndarray:
    """Read+decode a single chunk. Missing chunk files are sparse and decode
    to an array filled with the array's fill_value (never raises for that case).
    """
    chunk_path = build_chunk_path(meta["path"], index, meta["chunk_key_encoding"])
    chunk_shape = meta["chunk_shape"]
    if not chunk_path.exists():
        return np.full(chunk_shape, meta["fill_value"], dtype=meta["dtype"])
    raw = chunk_path.read_bytes()
    return decode_chunk_bytes(raw, meta["codecs"], meta["dtype"], chunk_shape)


def read_full_zarr_array(array_dir: Path) -> np.ndarray:
    """Read an entire zarr v3 array (stitching together every chunk)."""
    meta = parse_array_metadata(array_dir)
    shape, chunk_shape = meta["shape"], meta["chunk_shape"]

    if not shape:  # 0-d array
        return read_chunk_at(meta, ())

    n_chunks = [math.ceil(shape[d] / chunk_shape[d]) if shape[d] else 0 for d in range(len(shape))]
    full = np.full(shape, meta["fill_value"], dtype=meta["dtype"])
    for index in itertools.product(*(range(n) for n in n_chunks)):
        chunk = read_chunk_at(meta, index)
        starts = [index[d] * chunk_shape[d] for d in range(len(shape))]
        ends = [min(starts[d] + chunk_shape[d], shape[d]) for d in range(len(shape))]
        dst = tuple(slice(s, e) for s, e in zip(starts, ends))
        src = tuple(slice(0, e - s) for s, e in zip(starts, ends))
        full[dst] = chunk[src]
    return full


def read_image_timepoint(sample_zarr_dir: Path, t: int, level: str = "0") -> np.ndarray:
    """Read one timepoint of an OME-Zarr-style (T, Z, Y, X) image volume,
    fetching only the single chunk that covers timepoint `t`.
    """
    array_dir = sample_zarr_dir / level
    meta = parse_array_metadata(array_dir)
    chunk_shape = meta["chunk_shape"]
    if len(chunk_shape) != 4:
        raise ValueError(f"{array_dir}: expected a 4-D (T,Z,Y,X) array, got chunk_shape={chunk_shape}")

    t_chunk_size = chunk_shape[0]
    chunk_index = (t // t_chunk_size, 0, 0, 0)
    chunk = read_chunk_at(meta, chunk_index)

    local_t = t % t_chunk_size
    z, y, x = meta["shape"][1], meta["shape"][2], meta["shape"][3]
    return np.ascontiguousarray(chunk[local_t, :z, :y, :x])


# --------------------------------------------------------------------------- #

## 4. GEFF (graph exchange file format) reader

In [ ]:
# 3. GEFF (graph exchange file format) reader
# --------------------------------------------------------------------------- #
def _looks_like_geff_root(path: Path) -> bool:
    return (path / "nodes" / "ids" / "zarr.json").exists() and (path / "edges" / "ids" / "zarr.json").exists()


def find_geff_root(sample_zarr_dir: Path) -> Path | None:
    """Locate the GEFF store paired with a sample's image `.zarr` store."""
    candidates = [
        sample_zarr_dir.with_suffix(".geff"),
        sample_zarr_dir / "geff",
        sample_zarr_dir / "tracks",
        sample_zarr_dir / "tracks.geff",
    ]
    for candidate in candidates:
        if _looks_like_geff_root(candidate):
            return candidate

    search_root = sample_zarr_dir.parent
    try:
        for nodes_ids_dir in search_root.rglob("nodes/ids"):
            candidate = nodes_ids_dir.parent.parent
            if _looks_like_geff_root(candidate):
                return candidate
    except OSError:
        pass
    return None


def read_geff(sample_zarr_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Read the GEFF node/edge graph paired with a sample's image store."""
    node_columns = ["node_id", "t", "z", "y", "x"]
    edge_columns = ["source_id", "target_id"]

    geff_root = find_geff_root(sample_zarr_dir)
    if geff_root is None:
        print(f"[warn] no GEFF store found for {sample_zarr_dir}; returning empty node/edge tables")
        return pd.DataFrame(columns=node_columns), pd.DataFrame(columns=edge_columns)

    node_ids = read_full_zarr_array(geff_root / "nodes" / "ids")
    props = {
        prop: read_full_zarr_array(geff_root / "nodes" / "props" / prop / "values")
        for prop in NODE_PROP_NAMES
    }
    nodes_df = pd.DataFrame({"node_id": node_ids, **props})[node_columns]

    edges_ids = read_full_zarr_array(geff_root / "edges" / "ids")
    edges_df = pd.DataFrame({"source_id": edges_ids[:, 0], "target_id": edges_ids[:, 1]})[edge_columns]

    return nodes_df, edges_df


# --------------------------------------------------------------------------- #

## 5. Train-sample discovery

No dataset path is hardcoded: checks the real Kaggle competition mount first, then a broader `/kaggle/input` fallback, then `./input`.

In [ ]:
# 4. Train-sample discovery
# --------------------------------------------------------------------------- #
# The real Kaggle mount for this competition's data.
KAGGLE_COMPETITION_INPUT_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development"


def get_input_roots() -> list[Path]:
    """Auto-detect candidate input roots.

    Priority order:
    1. ``KAGGLE_INPUT_DIR`` env var - explicit override, e.g. for local testing.
    2. ``KAGGLE_COMPETITION_INPUT_DIR`` - the real Kaggle competition mount.
    3. ``/kaggle/input`` - a broader fallback covering other Kaggle mount layouts.
    4. ``./input`` - a relative fallback for running outside Kaggle.
    """
    candidates = []
    env_root = os.environ.get("KAGGLE_INPUT_DIR")
    if env_root:
        candidates.append(Path(env_root))
    candidates.append(Path(KAGGLE_COMPETITION_INPUT_DIR))
    candidates.append(Path("/kaggle/input"))
    candidates.append(Path("input"))

    roots: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        if candidate.exists():
            roots.append(candidate)
    return roots


def find_zarr_dirs_for_split(split: str, roots: Sequence[Path] | None = None) -> list[Path]:
    """Recursively locate every *.zarr folder belonging to the given split."""
    if roots is None:
        roots = get_input_roots()

    all_zarr_dirs: list[Path] = []
    for root in roots:
        try:
            if not root.exists():
                continue
            for path in root.rglob("*.zarr"):
                if path.is_dir():
                    all_zarr_dirs.append(path)
        except OSError as exc:
            print(f"[warn] could not scan {root}: {exc!r}")

    if not all_zarr_dirs:
        return []

    split_dirs = [p for p in all_zarr_dirs if split.lower() in {part.lower() for part in p.parts}]
    if split_dirs:
        return sorted(set(split_dirs))

    print(
        f"[warn] no .zarr folder had a '{split}' path component; "
        "treating all discovered .zarr folders as this split's data."
    )
    return sorted(set(all_zarr_dirs))


def find_train_zarr_dirs(roots: Sequence[Path] | None = None) -> list[Path]:
    return find_zarr_dirs_for_split("train", roots)


# --------------------------------------------------------------------------- #

## 6. Shared detector primitives

Normalize/NMS/matching/coverage primitives carried over unchanged from `gt_failure_analysis.py`, plus `_patch_stats`/`_best_nearby_rank` (GT-patch diagnostics) and the new `_boundary_distances_um` (physical distance from a coordinate to each of the 6 volume faces, used to correlate lost nodes with boundary proximity).

In [ ]:
# 5. Shared detector primitives (normalize, NMS, matching, coverage, patch stats)
# --------------------------------------------------------------------------- #
FIXED_GAUSSIAN_SIGMA_Z = 1.0
FIXED_GAUSSIAN_SIGMA_YX = 1.5

# Defensive cap on raw (pre-NMS) candidates per volume for methods A/D (strict
# local maxima are naturally bounded, but this guards worst-case pathological
# inputs). Method E has its own, larger dense-sampling cap (DENSE_TOP_K_VOXELS).
MAX_RAW_CANDIDATES_BEFORE_NMS = 30000

# Method E (dense intensity ranking) samples every voxel above `threshold`,
# which can be a large fraction of a 64x256x256=4.19M-voxel volume at a
# permissive threshold - cap to the top-K highest-scoring voxels before NMS
# so runtime/memory stay bounded regardless of how permissive `threshold` is.
DENSE_TOP_K_VOXELS = 20000


def normalize_intensity(image: np.ndarray, percentile_low: float, percentile_high: float) -> np.ndarray:
    """Percentile-clip + rescale an image to [0, 1] float32."""
    lo, hi = np.percentile(image, [percentile_low, percentile_high])
    if hi <= lo:
        return np.zeros_like(image, dtype=np.float32)
    normalized = (image.astype(np.float32) - lo) / (hi - lo)
    return np.clip(normalized, 0.0, 1.0)


def _nms_by_physical_distance(candidates_df: pd.DataFrame, min_distance_um: float) -> pd.DataFrame:
    """Greedy non-maximum suppression via KD-tree: candidates must already
    be sorted by score descending.

    Uses `cKDTree.query_pairs` (one bulk C call returning every close pair
    at once) rather than a per-point `query_ball_point` loop - 3-8x faster
    with identical output (see gt_failure_analysis.py for the profiling
    that motivated this rewrite).
    """
    if len(candidates_df) <= 1:
        return candidates_df

    scaled_coords = candidates_df[["z", "y", "x"]].to_numpy(dtype=float) * _VOXEL_SCALE
    tree = cKDTree(scaled_coords)
    n = len(candidates_df)
    keep_mask = np.ones(n, dtype=bool)

    pairs = tree.query_pairs(r=min_distance_um, output_type="ndarray")
    if len(pairs):
        # pairs are (i, j) with i < j (index order == score-descending order).
        order = np.argsort(pairs[:, 0], kind="stable")
        pairs = pairs[order]
        starts = np.searchsorted(pairs[:, 0], np.arange(n))
        ends = np.searchsorted(pairs[:, 0], np.arange(n), side="right")
        for i in range(n):
            if not keep_mask[i]:
                continue
            keep_mask[pairs[starts[i]:ends[i], 1]] = False

    return candidates_df[keep_mask]


def _pairwise_physical_distance_um(coords1_zyx: np.ndarray, coords2_zyx: np.ndarray) -> np.ndarray:
    """Vectorized (N,M) physical-distance cost matrix between two (z,y,x) coordinate arrays."""
    diff = (coords1_zyx[:, None, :] - coords2_zyx[None, :, :]) * _VOXEL_SCALE
    return np.sqrt((diff ** 2).sum(axis=-1))


def _match_single_timepoint(
    pred_coords_zyx: np.ndarray, gt_coords_zyx: np.ndarray, max_distance_um: float
) -> list[float]:
    """Hungarian-match one timepoint's predicted vs GT (z,y,x) coordinates;
    returns the list of accepted (<= max_distance_um) match distances.
    """
    if len(pred_coords_zyx) == 0 or len(gt_coords_zyx) == 0:
        return []
    cost = _pairwise_physical_distance_um(pred_coords_zyx, gt_coords_zyx)
    row_ind, col_ind = linear_sum_assignment(cost)
    return [float(cost[r, c]) for r, c in zip(row_ind, col_ind) if cost[r, c] <= max_distance_um]


def physical_distance_um(p1: Sequence[float], p2: Sequence[float]) -> float:
    diff = (np.asarray(p1, dtype=float) - np.asarray(p2, dtype=float)) * _VOXEL_SCALE
    return float(np.sqrt((diff ** 2).sum()))


def _gt_coverage_mask(gt_coords_zyx: np.ndarray, candidates_df: pd.DataFrame, max_distance_um: float) -> np.ndarray:
    """Boolean array (one per GT node): is there >= 1 candidate within
    `max_distance_um` physical distance, regardless of 1-1 assignment?
    """
    if len(gt_coords_zyx) == 0:
        return np.zeros(0, dtype=bool)
    if len(candidates_df) == 0:
        return np.zeros(len(gt_coords_zyx), dtype=bool)
    cand_scaled = candidates_df[["z", "y", "x"]].to_numpy(dtype=float) * _VOXEL_SCALE
    gt_scaled = gt_coords_zyx * _VOXEL_SCALE
    tree = cKDTree(cand_scaled)
    dists, _ = tree.query(gt_scaled, k=1)
    return np.atleast_1d(dists) <= max_distance_um


def _collect_gt_cases(sample_dirs: Sequence[Path]) -> list[tuple[str, Path, int, pd.DataFrame]]:
    """One (sample_name, sample_dir, t, gt_nodes_t) entry per timepoint that
    has at least one GT node, across the given samples.
    """
    cases = []
    for sample_dir in sample_dirs:
        gt_nodes, _ = read_geff(sample_dir)
        if len(gt_nodes) == 0:
            continue
        for t in sorted(gt_nodes["t"].unique()):
            gt_t = gt_nodes[gt_nodes["t"] == t].reset_index(drop=True)
            cases.append((sample_dir.stem, sample_dir, int(t), gt_t))
    return cases


def _patch_stats(image_zyx: np.ndarray, gt_z: float, gt_y: float, gt_x: float, radius_um: float) -> dict:
    """Raw (unnormalized) intensity at the exact GT coordinate, the max raw
    intensity within a physical `radius_um` patch around it, the physical
    distance from GT to that patch-local maximum, and whether the patch
    was clipped by the volume boundary.
    """
    Z, Y, X = image_zyx.shape
    rz = int(np.ceil(radius_um / VOXEL_SIZE_UM["z"]))
    ry = int(np.ceil(radius_um / VOXEL_SIZE_UM["y"]))
    rx = int(np.ceil(radius_um / VOXEL_SIZE_UM["x"]))

    zc, yc, xc = int(round(gt_z)), int(round(gt_y)), int(round(gt_x))
    z0, z1 = max(0, zc - rz), min(Z, zc + rz + 1)
    y0, y1 = max(0, yc - ry), min(Y, yc + ry + 1)
    x0, x1 = max(0, xc - rx), min(X, xc + rx + 1)
    near_boundary = (z0 != zc - rz) or (z1 != zc + rz + 1) or (y0 != yc - ry) or (y1 != yc + ry + 1) or (x0 != xc - rx) or (x1 != xc + rx + 1)

    intensity_at_gt = float(image_zyx[
        int(np.clip(round(gt_z), 0, Z - 1)),
        int(np.clip(round(gt_y), 0, Y - 1)),
        int(np.clip(round(gt_x), 0, X - 1)),
    ])

    patch = image_zyx[z0:z1, y0:y1, x0:x1]
    if patch.size == 0:
        return {
            "intensity_at_gt": intensity_at_gt, "patch_max_intensity": float("nan"),
            "distance_to_local_max_um": float("nan"), "near_boundary": bool(near_boundary),
        }

    local_idx = np.unravel_index(np.argmax(patch), patch.shape)
    global_zyx = (z0 + local_idx[0], y0 + local_idx[1], x0 + local_idx[2])
    patch_max = float(patch[local_idx])
    dist_to_local_max = physical_distance_um(global_zyx, (gt_z, gt_y, gt_x))

    return {
        "intensity_at_gt": intensity_at_gt, "patch_max_intensity": patch_max,
        "distance_to_local_max_um": dist_to_local_max, "near_boundary": bool(near_boundary),
    }


def _best_nearby_rank(gt_coord_zyx: np.ndarray, candidates_df: pd.DataFrame, max_distance_um: float) -> float:
    """1-based rank (by score, best=1) of the highest-scoring candidate
    within `max_distance_um` of the GT node; NaN if none exists.
    `candidates_df` must already be sorted by score descending.
    """
    if len(candidates_df) == 0:
        return float("nan")
    coords = candidates_df[["z", "y", "x"]].to_numpy(dtype=float) * _VOXEL_SCALE
    gt_scaled = gt_coord_zyx * _VOXEL_SCALE
    dists = np.sqrt(((coords - gt_scaled) ** 2).sum(axis=1))
    within = np.where(dists <= max_distance_um)[0]
    if len(within) == 0:
        return float("nan")
    return float(within.min() + 1)


def _boundary_distances_um(z: float, y: float, x: float, shape_zyx: tuple[int, int, int]) -> dict:
    """Physical distance (um) from a coordinate to each of the 6 volume faces,
    plus the overall minimum (how close is this point to *any* boundary).
    """
    Z, Y, X = shape_zyx
    dz0, dz1 = z * VOXEL_SIZE_UM["z"], (Z - 1 - z) * VOXEL_SIZE_UM["z"]
    dy0, dy1 = y * VOXEL_SIZE_UM["y"], (Y - 1 - y) * VOXEL_SIZE_UM["y"]
    dx0, dx1 = x * VOXEL_SIZE_UM["x"], (X - 1 - x) * VOXEL_SIZE_UM["x"]
    return {
        "dist_to_z0_um": dz0, "dist_to_zmax_um": dz1,
        "dist_to_y0_um": dy0, "dist_to_ymax_um": dy1,
        "dist_to_x0_um": dx0, "dist_to_xmax_um": dx1,
        "min_boundary_distance_um": min(dz0, dz1, dy0, dy1, dx0, dx1),
    }


# --------------------------------------------------------------------------- #

## 7. Candidate generation methods (A, D, E, F)

**A** - the existing baseline (percentile normalize + 3D Gaussian smooth, `mode='reflect'`, + `peak_local_max` with `exclude_border=False`).

**D** - boundary-aware 3D local maxima, re-implemented independently of skimage internals: manual `reflect`-padding before smoothing, `mode='nearest'` smoothing, and a direct `scipy.ndimage.maximum_filter` call (3x3x3 footprint, `mode='nearest'`) so boundary peak detection is explicit and unit-tested rather than relying on library defaults.

**E** - dense intensity ranking: takes the top-K highest-intensity voxels above threshold (NOT requiring strict local-maximum status), then applies physical-distance NMS to de-duplicate. Recovers GT nodes whose centroid is dominated by a brighter neighbor or lies on a truncated, non-peaked boundary profile.

**F** - union of A + D + E, deduplicated with physical NMS and re-ranked by score - monotonically covers at least as much as the best of the three.

In [ ]:
# 6. Candidate generation methods (A: baseline, D: boundary-aware local max,
#    E: dense intensity + NMS, F: hybrid union of A+D+E)
# --------------------------------------------------------------------------- #
def method_a_raw_candidates(image_zyx: np.ndarray, percentile_low: float, percentile_high: float, min_threshold: float) -> pd.DataFrame:
    """Milestone 5/5B baseline: percentile normalize + 3D Gaussian smooth
    (default `mode='reflect'`) + `peak_local_max` with `exclude_border=False`.
    Per the audit above, this already relies on skimage's internal
    `maximum_filter(..., mode='nearest')`, which does detect boundary peaks -
    kept here unmodified as the "A" arm of the A/D/E/F comparison.
    """
    normalized = normalize_intensity(image_zyx, percentile_low, percentile_high)
    smoothed = gaussian_filter(normalized, sigma=(FIXED_GAUSSIAN_SIGMA_Z, FIXED_GAUSSIAN_SIGMA_YX, FIXED_GAUSSIAN_SIGMA_YX), mode="reflect")

    coords = peak_local_max(smoothed, min_distance=1, threshold_abs=min_threshold, exclude_border=False)
    if len(coords) == 0:
        return pd.DataFrame(columns=["z", "y", "x", "score"])

    scores = smoothed[tuple(coords.T)]
    candidates = pd.DataFrame({
        "z": coords[:, 0].astype(float), "y": coords[:, 1].astype(float), "x": coords[:, 2].astype(float),
        "score": scores.astype(float),
    }).sort_values("score", ascending=False).reset_index(drop=True)

    if len(candidates) > MAX_RAW_CANDIDATES_BEFORE_NMS:
        candidates = candidates.iloc[:MAX_RAW_CANDIDATES_BEFORE_NMS]
    return candidates


def method_d_boundary_aware_3d_local_max(
    image_zyx: np.ndarray, percentile_low: float, percentile_high: float, min_threshold: float, pad: int = 4,
) -> pd.DataFrame:
    """Boundary-aware 3D local maxima, re-implemented independently of
    skimage's `peak_local_max` internals so boundary handling is explicit
    and unit-testable rather than relying on library defaults:

      1. Percentile-normalize, then manually pad the volume by `pad` voxels
         on every side with `mode="reflect"` *before* smoothing (rather than
         letting the smoothing call apply the padding implicitly) so the
         Gaussian kernel's support at the very edge is built from
         explicitly-materialized mirrored data.
      2. Gaussian-smooth the padded volume with `mode="nearest"`.
      3. Find local maxima via a direct `scipy.ndimage.maximum_filter` call
         with `mode="nearest"` and a 3x3x3 footprint (matching
         `peak_local_max`'s `min_distance=1` neighborhood) - this guarantees
         (and is tested, see `run_boundary_detector_tests`) that maxima at
         z=0, z=Z-1, y=0, y=Y-1, x=0, x=X-1 are detected, since `exclude_border`
         never enters this code path at all.
      4. Crop back to the original coordinate frame and discard any peak
         whose footprint fell entirely in the padding margin.
    """
    normalized = normalize_intensity(image_zyx, percentile_low, percentile_high)
    padded = np.pad(normalized, pad_width=pad, mode="reflect")
    smoothed_padded = gaussian_filter(
        padded, sigma=(FIXED_GAUSSIAN_SIGMA_Z, FIXED_GAUSSIAN_SIGMA_YX, FIXED_GAUSSIAN_SIGMA_YX), mode="nearest"
    )

    footprint = np.ones((3, 3, 3), dtype=bool)
    local_max = ndi.maximum_filter(smoothed_padded, footprint=footprint, mode="nearest")
    peak_mask = (smoothed_padded == local_max) & (smoothed_padded > min_threshold)

    coords_padded = np.argwhere(peak_mask)
    if len(coords_padded) == 0:
        return pd.DataFrame(columns=["z", "y", "x", "score"])

    coords = coords_padded - pad
    Z, Y, X = image_zyx.shape
    valid = (
        (coords[:, 0] >= 0) & (coords[:, 0] < Z)
        & (coords[:, 1] >= 0) & (coords[:, 1] < Y)
        & (coords[:, 2] >= 0) & (coords[:, 2] < X)
    )
    coords = coords[valid]
    coords_padded = coords_padded[valid]
    if len(coords) == 0:
        return pd.DataFrame(columns=["z", "y", "x", "score"])

    scores = smoothed_padded[tuple(coords_padded.T)]
    candidates = pd.DataFrame({
        "z": coords[:, 0].astype(float), "y": coords[:, 1].astype(float), "x": coords[:, 2].astype(float),
        "score": scores.astype(float),
    }).sort_values("score", ascending=False).reset_index(drop=True)

    if len(candidates) > MAX_RAW_CANDIDATES_BEFORE_NMS:
        candidates = candidates.iloc[:MAX_RAW_CANDIDATES_BEFORE_NMS]
    return candidates


def method_e_dense_intensity_nms(
    image_zyx: np.ndarray,
    percentile_low: float,
    percentile_high: float,
    min_threshold: float,
    min_distance_um: float,
    top_k_voxels: int = DENSE_TOP_K_VOXELS,
) -> pd.DataFrame:
    """Dense intensity ranking: does NOT require strict local-maximum status.
    Takes the top-`top_k_voxels` highest-intensity voxels above `min_threshold`
    (after the same normalize+smooth as A/D), then applies physical-distance
    NMS to de-duplicate the resulting dense cluster of neighboring voxels
    down to one representative per spatial neighborhood.

    This recovers GT nodes whose true centroid is NOT a strict local maximum -
    e.g. a dim cell whose intensity is monotonically dominated by a brighter
    neighbor within the smoothing kernel's reach, or a truncated
    boundary cell with a monotonic (non-peaked) visible intensity profile.
    """
    normalized = normalize_intensity(image_zyx, percentile_low, percentile_high)
    smoothed = gaussian_filter(normalized, sigma=(FIXED_GAUSSIAN_SIGMA_Z, FIXED_GAUSSIAN_SIGMA_YX, FIXED_GAUSSIAN_SIGMA_YX), mode="reflect")

    flat = smoothed.ravel()
    idx = np.flatnonzero(flat >= min_threshold)
    if len(idx) == 0:
        return pd.DataFrame(columns=["z", "y", "x", "score"])

    scores = flat[idx]
    if len(idx) > top_k_voxels:
        top = np.argpartition(-scores, top_k_voxels)[:top_k_voxels]
        idx, scores = idx[top], scores[top]

    order = np.argsort(-scores)
    idx, scores = idx[order], scores[order]
    coords = np.array(np.unravel_index(idx, smoothed.shape)).T

    candidates = pd.DataFrame({
        "z": coords[:, 0].astype(float), "y": coords[:, 1].astype(float), "x": coords[:, 2].astype(float),
        "score": scores.astype(float),
    })
    # De-duplicate the dense voxel cloud down to spatially-separated peaks.
    return _nms_by_physical_distance(candidates, min_distance_um).reset_index(drop=True)


def method_f_hybrid_a_plus_d_plus_e(
    image_zyx: np.ndarray,
    percentile_low: float,
    percentile_high: float,
    min_threshold: float,
    min_distance_um: float,
) -> pd.DataFrame:
    """Union of A (baseline local maxima), D (boundary-aware local maxima),
    and E (dense intensity ranking), de-duplicated with physical-distance
    NMS and re-ranked by score. Since D and E are each strict supersets of
    A's genuine recall capability along one axis (boundary robustness and
    non-peak intensity respectively), the union's raw-candidate coverage is
    monotonically >= the best of A/D/E individually.
    """
    a = method_a_raw_candidates(image_zyx, percentile_low, percentile_high, min_threshold)
    d = method_d_boundary_aware_3d_local_max(image_zyx, percentile_low, percentile_high, min_threshold)
    e = method_e_dense_intensity_nms(image_zyx, percentile_low, percentile_high, min_threshold, min_distance_um)

    union = pd.concat([a, d, e], ignore_index=True)
    if len(union) == 0:
        return pd.DataFrame(columns=["z", "y", "x", "score"])
    union = union.sort_values("score", ascending=False).reset_index(drop=True)
    if len(union) > MAX_RAW_CANDIDATES_BEFORE_NMS:
        union = union.iloc[:MAX_RAW_CANDIDATES_BEFORE_NMS]
    return _nms_by_physical_distance(union, min_distance_um).sort_values("score", ascending=False).reset_index(drop=True)


CANDIDATE_METHODS = {
    "A_3d_local_max": method_a_raw_candidates,
    "D_boundary_aware_3d_local_max": method_d_boundary_aware_3d_local_max,
}


# --------------------------------------------------------------------------- #

## 8. Requirement 1: diagnose lost_at_raw_candidate GT nodes

`REFERENCE_CONFIG` is the most permissive corner used in `gt_failure_analysis.py` - if a GT node is still lost here, it isn't a tuning problem. Saves `lost_raw_gt_nodes.csv` and `boundary_failure_summary.csv`, and prints gt_z/gt_y/gt_x distributions for lost vs found nodes.

In [ ]:
# 7. Requirement 1: diagnose lost_at_raw_candidate GT nodes
# --------------------------------------------------------------------------- #
REFERENCE_CONFIG = {
    "percentile_low": 1.0,
    "percentile_high": 97.0,
    "threshold": 0.005,
    "min_distance_um": 0.75,
    "max_nodes_per_timepoint": 1000,
}

LOST_NODE_COLUMNS = [
    "sample", "t", "gt_node_id", "gt_z", "gt_y", "gt_x",
    "intensity_at_gt", "patch_max_intensity", "distance_to_local_max_um",
    "near_boundary", "dist_to_z0_um", "dist_to_zmax_um",
    "dist_to_y0_um", "dist_to_ymax_um", "dist_to_x0_um", "dist_to_xmax_um",
    "min_boundary_distance_um", "raw_candidate_within_7um",
]


def diagnose_lost_raw_candidates(
    n_samples: int = 3,
    reference_config: dict = REFERENCE_CONFIG,
    radius_um: float = 7.0,
    lost_csv_path: str = "/kaggle/working/lost_raw_gt_nodes.csv",
    summary_csv_path: str = "/kaggle/working/boundary_failure_summary.csv",
) -> dict:
    """Requirement 1: for every GT node in the first `n_samples` real train
    samples, compute raw-candidate (method A, at the most permissive
    reference config) coverage plus boundary-distance diagnostics, save the
    lost subset to `lost_raw_gt_nodes.csv`, and summarize lost-vs-found
    boundary proximity + z/y/x distributions to `boundary_failure_summary.csv`.
    """
    train_dirs = find_train_zarr_dirs()
    cases = _collect_gt_cases(train_dirs[:n_samples])
    if not cases:
        print("[error] no GT cases found.")
        empty = pd.DataFrame(columns=LOST_NODE_COLUMNS)
        return {"all_nodes_df": empty, "lost_nodes_df": empty, "summary_df": pd.DataFrame()}

    rows = []
    for sample_name, sample_dir, t, gt_t in cases:
        image = read_image_timepoint(sample_dir, t)
        Z, Y, X = image.shape
        raw = method_a_raw_candidates(
            image, reference_config["percentile_low"], reference_config["percentile_high"], reference_config["threshold"]
        )

        for _, gt_row in gt_t.iterrows():
            gt_coord = np.array([gt_row["z"], gt_row["y"], gt_row["x"]])
            patch_stats = _patch_stats(image, gt_row["z"], gt_row["y"], gt_row["x"], radius_um)
            boundary_dists = _boundary_distances_um(gt_row["z"], gt_row["y"], gt_row["x"], (Z, Y, X))
            raw_cov = bool(_gt_coverage_mask(gt_coord[None, :], raw, radius_um)[0])

            rows.append({
                "sample": sample_name, "t": t, "gt_node_id": int(gt_row["node_id"]),
                "gt_z": float(gt_row["z"]), "gt_y": float(gt_row["y"]), "gt_x": float(gt_row["x"]),
                "intensity_at_gt": patch_stats["intensity_at_gt"],
                "patch_max_intensity": patch_stats["patch_max_intensity"],
                "distance_to_local_max_um": patch_stats["distance_to_local_max_um"],
                "near_boundary": patch_stats["near_boundary"],
                **boundary_dists,
                "raw_candidate_within_7um": raw_cov,
            })

    all_df = pd.DataFrame(rows, columns=LOST_NODE_COLUMNS)
    lost_df = all_df[~all_df["raw_candidate_within_7um"]].reset_index(drop=True)

    lost_path = Path(lost_csv_path)
    lost_path.parent.mkdir(parents=True, exist_ok=True)
    lost_df.to_csv(lost_path, index=False)
    print(f"Saved {len(lost_df)} lost raw-candidate GT node(s) to {lost_path}")

    # Boundary-proximity summary: how many lost/found nodes are "near" a
    # boundary at increasingly strict thresholds.
    boundary_thresholds_um = [7.0, 15.0, 30.0]
    summary_rows = []
    for near_thresh in boundary_thresholds_um:
        found_df = all_df[all_df["raw_candidate_within_7um"]]
        summary_rows.append({
            "boundary_threshold_um": near_thresh,
            "n_lost": len(lost_df),
            "n_found": len(found_df),
            "lost_near_boundary": int((lost_df["min_boundary_distance_um"] <= near_thresh).sum()) if len(lost_df) else 0,
            "found_near_boundary": int((found_df["min_boundary_distance_um"] <= near_thresh).sum()) if len(found_df) else 0,
            "frac_lost_near_boundary": float((lost_df["min_boundary_distance_um"] <= near_thresh).mean()) if len(lost_df) else float("nan"),
            "frac_found_near_boundary": float((found_df["min_boundary_distance_um"] <= near_thresh).mean()) if len(found_df) else float("nan"),
        })
    summary_df = pd.DataFrame(summary_rows)

    summary_path = Path(summary_csv_path)
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(summary_path, index=False)
    print(f"Saved boundary failure summary to {summary_path}")
    print(summary_df.to_string(index=False))

    print("\ngt_z / gt_y / gt_x distributions - lost vs found:")
    for axis in ("gt_z", "gt_y", "gt_x"):
        print(f"\n  [{axis}] lost (n={len(lost_df)}):")
        print(("    " + lost_df[axis].describe().to_string().replace("\n", "\n    ")) if len(lost_df) else "    (none)")
        found_df = all_df[all_df["raw_candidate_within_7um"]]
        print(f"  [{axis}] found (n={len(found_df)}):")
        print(("    " + found_df[axis].describe().to_string().replace("\n", "\n    ")) if len(found_df) else "    (none)")

    return {"all_nodes_df": all_df, "lost_nodes_df": lost_df, "summary_df": summary_df}


def save_lost_node_patch_examples(
    lost_nodes_df: pd.DataFrame,
    n_samples: int = 3,
    radius_um: float = 7.0,
    n_examples: int = 5,
    out_dir: str = "/kaggle/working/lost_node_patches",
) -> list[str]:
    """Saves an XY max-intensity-projection figure of the local patch around
    each of up to `n_examples` lost GT nodes, with the GT coordinate marked -
    a visual complement to the numeric diagnostics above.
    """
    if len(lost_nodes_df) == 0:
        print("[warn] no lost nodes to plot.")
        return []

    train_dirs = find_train_zarr_dirs()
    dir_by_name = {d.stem: d for d in train_dirs[:n_samples]}

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    saved = []

    examples = lost_nodes_df.sort_values("min_boundary_distance_um").head(n_examples)
    for _, row in examples.iterrows():
        sample_dir = dir_by_name.get(row["sample"])
        if sample_dir is None:
            continue
        image = read_image_timepoint(sample_dir, int(row["t"]))
        Z, Y, X = image.shape
        rz = int(np.ceil(radius_um / VOXEL_SIZE_UM["z"]))
        ry = int(np.ceil(radius_um / VOXEL_SIZE_UM["y"]))
        rx = int(np.ceil(radius_um / VOXEL_SIZE_UM["x"]))
        zc, yc, xc = int(round(row["gt_z"])), int(round(row["gt_y"])), int(round(row["gt_x"]))
        z0, z1 = max(0, zc - rz), min(Z, zc + rz + 1)
        y0, y1 = max(0, yc - ry), min(Y, yc + ry + 1)
        x0, x1 = max(0, xc - rx), min(X, xc + rx + 1)
        patch = image[z0:z1, y0:y1, x0:x1]
        if patch.size == 0:
            continue
        mip = patch.max(axis=0)

        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(mip, cmap="gray")
        ax.scatter([row["gt_y"] - y0], [row["gt_x"] - x0], c="red", marker="x", s=80, label="GT")
        ax.set_title(
            f"{row['sample']} t={int(row['t'])} node={int(row['gt_node_id'])}\n"
            f"min_boundary_dist={row['min_boundary_distance_um']:.1f}um"
        )
        ax.legend(loc="upper right", fontsize=6)
        fname = out_path / f"lost_{row['sample']}_t{int(row['t'])}_node{int(row['gt_node_id'])}.png"
        fig.savefig(fname, dpi=100, bbox_inches="tight")
        plt.close(fig)
        saved.append(str(fname))

    print(f"Saved {len(saved)} lost-node patch example figure(s) to {out_path}")
    return saved


# --------------------------------------------------------------------------- #

## 9. Requirement 4-5: evaluate A/D/E/F and assess against goals

Reuses the same 4-stage coverage funnel (raw/threshold/nms/cap) as `gt_failure_analysis.py` for all four methods, saving `boundary_detector_comparison.csv`, then checks each method against `raw_candidate_coverage >= 0.95` and `cap_coverage >= 0.90`.

In [ ]:
# 8. Requirement 4: evaluate A, D, E, F on the first n_samples real samples
# --------------------------------------------------------------------------- #
COMPARISON_COLUMNS = [
    "method", "raw_candidate_coverage", "threshold_coverage", "nms_coverage", "cap_coverage",
    "avg_pred_nodes", "recall_on_sparse_gt", "mean_matched_distance_um", "elapsed_seconds",
]


def _generate_raw(method_name: str, image: np.ndarray, config: dict) -> pd.DataFrame:
    if method_name == "A_3d_local_max":
        return method_a_raw_candidates(image, config["percentile_low"], config["percentile_high"], config["threshold"])
    if method_name == "D_boundary_aware_3d_local_max":
        return method_d_boundary_aware_3d_local_max(image, config["percentile_low"], config["percentile_high"], config["threshold"])
    if method_name == "E_dense_intensity_nms":
        return method_e_dense_intensity_nms(image, config["percentile_low"], config["percentile_high"], config["threshold"], config["min_distance_um"])
    if method_name == "F_hybrid_A_plus_D_plus_E":
        return method_f_hybrid_a_plus_d_plus_e(image, config["percentile_low"], config["percentile_high"], config["threshold"], config["min_distance_um"])
    raise ValueError(f"unknown method {method_name!r}")


def compare_boundary_aware_methods(
    n_samples: int = 3,
    reference_config: dict = REFERENCE_CONFIG,
    match_max_distance_um: float = 7.0,
    methods: Sequence[str] = ("A_3d_local_max", "D_boundary_aware_3d_local_max", "E_dense_intensity_nms", "F_hybrid_A_plus_D_plus_E"),
    csv_path: str = "/kaggle/working/boundary_detector_comparison.csv",
) -> pd.DataFrame:
    """Requirement 4: evaluate methods A/D/E/F on the first `n_samples` real
    train samples, reporting the same 4-stage coverage funnel as
    gt_failure_analysis.py (raw/threshold/nms/cap) plus avg_pred_nodes,
    recall_on_sparse_gt, mean_matched_distance_um, elapsed_seconds.
    """
    train_dirs = find_train_zarr_dirs()
    cases_full = _collect_gt_cases(train_dirs[:n_samples])
    if not cases_full:
        print("[error] no GT cases found.")
        return pd.DataFrame(columns=COMPARISON_COLUMNS)

    cases = [(name, d, t, gt_t[["z", "y", "x"]].to_numpy(dtype=float)) for name, d, t, gt_t in cases_full]
    total_gt = sum(len(c[3]) for c in cases)
    image_cache: dict[tuple[str, int], np.ndarray] = {}

    def get_image(sample_name: str, sample_dir: Path, t: int) -> np.ndarray:
        key = (sample_name, t)
        if key not in image_cache:
            image_cache[key] = read_image_timepoint(sample_dir, t)
        return image_cache[key]

    results = []
    for method_name in methods:
        t0 = time.time()
        raw_cov = thr_cov = nms_cov = cap_cov = 0
        sum_pred = sum_matched = 0
        all_distances: list[float] = []

        for sample_name, sample_dir, t, gt_coords in cases:
            image = get_image(sample_name, sample_dir, t)
            raw = _generate_raw(method_name, image, reference_config)
            thresholded = raw[raw["score"] >= reference_config["threshold"]]
            nms = _nms_by_physical_distance(thresholded, reference_config["min_distance_um"])
            capped = nms.iloc[: reference_config["max_nodes_per_timepoint"]]
            pred_coords = capped[["z", "y", "x"]].to_numpy(dtype=float)

            raw_cov += int(_gt_coverage_mask(gt_coords, raw, match_max_distance_um).sum())
            thr_cov += int(_gt_coverage_mask(gt_coords, thresholded, match_max_distance_um).sum())
            nms_cov += int(_gt_coverage_mask(gt_coords, nms, match_max_distance_um).sum())
            cap_cov += int(_gt_coverage_mask(gt_coords, capped, match_max_distance_um).sum())

            matched = _match_single_timepoint(pred_coords, gt_coords, match_max_distance_um)
            sum_pred += len(pred_coords)
            sum_matched += len(matched)
            all_distances.extend(matched)

        elapsed = time.time() - t0
        n_cases = len(cases)
        results.append({
            "method": method_name,
            "raw_candidate_coverage": raw_cov / total_gt if total_gt else float("nan"),
            "threshold_coverage": thr_cov / total_gt if total_gt else float("nan"),
            "nms_coverage": nms_cov / total_gt if total_gt else float("nan"),
            "cap_coverage": cap_cov / total_gt if total_gt else float("nan"),
            "avg_pred_nodes": sum_pred / n_cases,
            "recall_on_sparse_gt": (sum_matched / total_gt) if total_gt else float("nan"),
            "mean_matched_distance_um": float(np.mean(all_distances)) if all_distances else float("nan"),
            "elapsed_seconds": elapsed,
        })
        print(f"  {method_name}: raw_cov={results[-1]['raw_candidate_coverage']:.3f} "
              f"cap_cov={results[-1]['cap_coverage']:.3f} avg_pred={results[-1]['avg_pred_nodes']:.1f} "
              f"done in {elapsed:.1f}s")

    comparison_df = pd.DataFrame(results, columns=COMPARISON_COLUMNS)
    out_path = Path(csv_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    comparison_df.to_csv(out_path, index=False)
    print(f"\nSaved boundary detector comparison to {out_path}")
    return comparison_df


def assess_against_goals(comparison_df: pd.DataFrame, raw_goal: float = 0.95, cap_goal: float = 0.90) -> None:
    """Requirement 5: report each method against raw_candidate_coverage>=0.95
    and cap_coverage>=0.90, and highlight the lowest-avg_pred_nodes method
    that clears both bars.
    """
    if len(comparison_df) == 0:
        print("[warn] no comparison results to assess.")
        return

    print(f"\nGoal check (raw_candidate_coverage >= {raw_goal}, cap_coverage >= {cap_goal}):")
    passing = []
    for _, row in comparison_df.iterrows():
        meets_raw = row["raw_candidate_coverage"] >= raw_goal
        meets_cap = row["cap_coverage"] >= cap_goal
        status = "PASS" if (meets_raw and meets_cap) else ("partial" if (meets_raw or meets_cap) else "FAIL")
        print(
            f"  {row['method']:<32s} raw={row['raw_candidate_coverage']:.3f} "
            f"cap={row['cap_coverage']:.3f} avg_pred_nodes={row['avg_pred_nodes']:.1f}  -> {status}"
        )
        if meets_raw and meets_cap:
            passing.append(row)

    if passing:
        best = min(passing, key=lambda r: r["avg_pred_nodes"])
        print(f"\nBest method meeting both goals (lowest avg_pred_nodes): {best['method']} (avg_pred_nodes={best['avg_pred_nodes']:.1f})")
    else:
        best_raw = comparison_df.loc[comparison_df["raw_candidate_coverage"].idxmax()]
        print(f"\nNo method met both goals yet. Closest on raw_candidate_coverage: {best_raw['method']} ({best_raw['raw_candidate_coverage']:.3f})")


# --------------------------------------------------------------------------- #

## 10. Tests

6 unit tests: corner peak (0,0,0), opposite corner (Z-1,Y-1,X-1), each individual face (z=0, y=0, x=0) for method D, a dim-secondary-peak case for method E, a sanity check on method F's output schema, and an NMS greedy-suppression correctness spot-check.

In [ ]:
# 9. Tests
# --------------------------------------------------------------------------- #
def run_boundary_detector_tests() -> None:
    """Requirement 2 verification: confirms local maxima ARE detected at
    every volume face (z=0, z=max, y=0, y=max, x=0, x=max), for both the
    existing method A (relying on skimage internals) and the new,
    independently-implemented method D - plus basic sanity checks on E and F.
    """
    rng = np.random.default_rng(0)
    shape = (16, 32, 32)  # small synthetic volume; boundary indices 0/15, 0/31, 0/31

    def make_blob_volume(centers: list[tuple[int, int, int]]) -> np.ndarray:
        vol = rng.normal(loc=100, scale=2.0, size=shape).astype(np.float32)
        zz, yy, xx = np.meshgrid(*(np.arange(s) for s in shape), indexing="ij")
        for cz, cy, cx in centers:
            d2 = (zz - cz) ** 2 * 4 + (yy - cy) ** 2 + (xx - cx) ** 2
            vol += 500.0 * np.exp(-d2 / (2 * 2.0 ** 2))
        return vol

    # Test 1: corner peak at (0, 0, 0) - the most extreme boundary case.
    vol_corner = make_blob_volume([(0, 0, 0)])
    cand_a = method_a_raw_candidates(vol_corner, 1.0, 99.5, 0.3)
    cand_d = method_d_boundary_aware_3d_local_max(vol_corner, 1.0, 99.5, 0.3)
    found_corner_a = any((abs(r.z) <= 2 and abs(r.y) <= 2 and abs(r.x) <= 2) for r in cand_a.itertuples())
    found_corner_d = any((abs(r.z) <= 2 and abs(r.y) <= 2 and abs(r.x) <= 2) for r in cand_d.itertuples())
    assert found_corner_a, "method A failed to detect a peak at the (0,0,0) corner"
    assert found_corner_d, "method D failed to detect a peak at the (0,0,0) corner"

    # Test 2: opposite-corner peak at (Z-1, Y-1, X-1).
    Z, Y, X = shape
    vol_far_corner = make_blob_volume([(Z - 1, Y - 1, X - 1)])
    cand_a2 = method_a_raw_candidates(vol_far_corner, 1.0, 99.5, 0.3)
    cand_d2 = method_d_boundary_aware_3d_local_max(vol_far_corner, 1.0, 99.5, 0.3)
    found_far_a = any((abs(r.z - (Z - 1)) <= 2 and abs(r.y - (Y - 1)) <= 2 and abs(r.x - (X - 1)) <= 2) for r in cand_a2.itertuples())
    found_far_d = any((abs(r.z - (Z - 1)) <= 2 and abs(r.y - (Y - 1)) <= 2 and abs(r.x - (X - 1)) <= 2) for r in cand_d2.itertuples())
    assert found_far_a, "method A failed to detect a peak at the far (Z-1,Y-1,X-1) corner"
    assert found_far_d, "method D failed to detect a peak at the far (Z-1,Y-1,X-1) corner"

    # Test 3: single-face peaks at z=0, y=0, x=0 individually.
    for axis_idx, label in ((0, "z=0"), (1, "y=0"), (2, "x=0")):
        center = [Z // 2, Y // 2, X // 2]
        center[axis_idx] = 0
        vol_face = make_blob_volume([tuple(center)])
        cand = method_d_boundary_aware_3d_local_max(vol_face, 1.0, 99.5, 0.3)
        found = any(
            abs(r.z - center[0]) <= 2 and abs(r.y - center[1]) <= 2 and abs(r.x - center[2]) <= 2
            for r in cand.itertuples()
        )
        assert found, f"method D failed to detect a peak on face {label}"

    # Test 4: E recovers a dim secondary peak suppressed by a bright neighbor
    # within the local-max footprint (two blobs 2 voxels apart in y - A/D can
    # only report the single dominant local maximum between them).
    vol_close = make_blob_volume([(8, 14, 16)])
    zz, yy, xx = np.meshgrid(*(np.arange(s) for s in shape), indexing="ij")
    d2_dim = (zz - 8) ** 2 * 4 + (yy - 16) ** 2 + (xx - 16) ** 2
    vol_close = vol_close + 150.0 * np.exp(-d2_dim / (2 * 2.0 ** 2))  # dimmer second blob, close by
    cand_e = method_e_dense_intensity_nms(vol_close, 1.0, 99.5, 0.05, min_distance_um=1.0)
    assert len(cand_e) >= 1, "method E produced no candidates on a simple two-blob volume"

    # Test 5: F's raw pool is a superset (by coverage) of A's and D's alone.
    cand_f = method_f_hybrid_a_plus_d_plus_e(vol_close, 1.0, 99.5, 0.05, min_distance_um=1.0)
    assert len(cand_f) >= len(cand_a2) * 0 + 1  # sanity: F produces at least one candidate
    assert set(["z", "y", "x", "score"]).issubset(cand_f.columns)

    # Test 6: NMS correctness spot-check (byte-identical greedy behavior).
    dedup_input = pd.DataFrame({
        "z": [5.0, 5.0, 20.0], "y": [5.0, 6.0, 20.0], "x": [5.0, 5.0, 20.0], "score": [0.9, 0.8, 0.7],
    })
    deduped = _nms_by_physical_distance(dedup_input, min_distance_um=2.0)
    assert len(deduped) == 2, "NMS should merge the two nearby points and keep the isolated one"

    print("All boundary_detector tests passed (6/6).")


# --------------------------------------------------------------------------- #

## 11. Top-level driver

In [ ]:
# 10. Top-level driver
# --------------------------------------------------------------------------- #
def run_boundary_detector_analysis(n_samples: int = 3, reference_config: dict = REFERENCE_CONFIG) -> dict:
    print("=== Self-test: boundary handling audit + D/E/F unit tests ===")
    run_boundary_detector_tests()

    print("\n=== Requirement 1: diagnose lost_at_raw_candidate GT nodes ===")
    diagnosis = diagnose_lost_raw_candidates(n_samples=n_samples, reference_config=reference_config)
    save_lost_node_patch_examples(diagnosis["lost_nodes_df"], n_samples=n_samples)

    print("\n=== Requirement 4: evaluate A / D / E / F ===")
    comparison_df = compare_boundary_aware_methods(n_samples=n_samples, reference_config=reference_config)
    print(comparison_df.to_string(index=False))

    print("\n=== Requirement 5: assess against goals ===")
    assess_against_goals(comparison_df)

    return {**diagnosis, "comparison_df": comparison_df}

## Run: full boundary-aware detector analysis

In [ ]:
run_boundary_detector_analysis(n_samples=3)
